# Chat via LangGraph

In [1]:
import { readFileSync } from 'node:fs';
import { START, END, StateGraph } from "@langchain/langgraph";
import { Annotations } from "@common/annotations.ts";
import { getLogger } from '../../server/src/Logger.ts';
import { StateInfo, responseContent } from '../../server/src/agents/agent.ts';
import { encodeAnnotationsNode } from "../../server/src/agents/nodes/encodeAnnotationsNode.ts";
import { modelNode } from "../../server/src/agents/nodes/modelNode.ts";
import { HumanMessage, SystemMessage } from "@langchain/core/messages";

export const annNode = (state: typeof StateInfo.State) => {
  const PROMPT = readFileSync('../../server/src/agents/nodes/annotateNodePromptBroad.txt', 'utf-8');
  state.logger.info(`System Message:\n${PROMPT}`);
  state.logger.info(`Human Message:\n${state.systemData}`);
  return { messages: [
    new SystemMessage(PROMPT),
    new HumanMessage(state.systemData)
  ]};
}

// Define a new graph
const workflow = new StateGraph(StateInfo)
  .addNode("encodeAnnotations", encodeAnnotationsNode)
  .addNode("annotate", annNode)
  .addNode("model", modelNode)
  .addEdge(START, "encodeAnnotations")
  .addEdge("encodeAnnotations","annotate")
  .addEdge("annotate", "model")
  .addEdge("model", END);

// Compile graph
export const graph = workflow.compile();

const logger = getLogger();
const userUUID: string = "0";
const lhsText = readFileSync('../../frontend/public/data/AES/selected-text.txt', 'utf-8');
const rhsText = readFileSync('../../frontend/public/data/AES/pre-written.txt', 'utf-8');
const currentAnnotations: Annotations = { "mappings": [], "lhsLabels": [], "rhsLabels": [] };

const config = { configurable: { thread_id: userUUID } };
const output = await graph.invoke({ lhsText: lhsText, rhsText: rhsText, currentAnnotations: currentAnnotations, logger: logger }, config);
console.log(responseContent(output));


I'll help you generate annotations for the AES implementation. I'll focus on key aspects of the algorithm and its translation from the LHS text to the RHS text.

### CHAIN-OF-THOUGHT ANALYSIS

I'll analyze the texts and create annotations that capture their relationships, focusing on:
1. Algorithm overview
2. Key transformations
3. Cipher and inverse cipher functions
4. Key expansion
5. Specific transformations like SubBytes, ShiftRows, MixColumns

### JSON ANNOTATIONS

```json
[
{
  "description": "AES Algorithm Overview",
  "lhsText": ["The general function for executing AES-128, AES-192, or AES-256 is denoted by CIPHER(); its inverse is denoted by INVCIPHER()."],
  "rhsText": ["def AES128 (input : Array UInt8) (key : Array UInt8) : Array UInt8 :=
  let Nr := 10
  let Nk := 4
  let keySchedule := keyExpansion key Nk Nr
  cipher input keySchedule Nr"],
  "status": "success",
  "comment": ""
},
{
  "description": "Key Expansion Function",
  "lhsText": ["An expansion routine, denoted by